In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Au fost detectate {len(gpus)} GPU-uri active:")
    for i, gpu in enumerate(gpus):
        print(f" Placa {i}: {gpu.name}")
else:
    print("Niciun GPU detectat. Sistemul rulează lent pe CPU.")

Au fost detectate 2 GPU-uri active:
 Placa 0: /physical_device:GPU:0
 Placa 1: /physical_device:GPU:1


In [13]:
!pip install ai_edge_litert
from ai_edge_litert.interpreter import Interpreter

In [4]:
strategy = tf.distribute.MirroredStrategy()
print(f"Numărul de unități grafice sincronizate: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Numărul de unități grafice sincronizate: 2


In [5]:
from tensorflow import keras

modele = [
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/mobilenet_gpu_fer-2013-augmented.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_ck.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_fer-2013-original.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vrescnn_gpu_raf-db.keras"
]

for path in modele:
    model = keras.models.load_model(path)
    print(f"{path.split('/')[-1]} → input shape: {model.input_shape}")

mobilenet_gpu_fer-2013-augmented.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_ck.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_fer-2013-original.keras → input shape: (None, 48, 48, 1)
vrescnn_gpu_raf-db.keras → input shape: (None, 48, 48, 1)


In [47]:
from tensorflow import keras
from tensorflow.keras.models import load_model
import tensorflow as tf

model = load_model('/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vrescnn_gpu_raf-db.keras')   
model.export('some', format="tf_saved_model")

mod=tf.saved_model.load('some')

INFO:tensorflow:Assets written to: some/assets


INFO:tensorflow:Assets written to: some/assets


Saved artifact at 'some'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 48, 48, 1), dtype=tf.float32, name='input_layer')]
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  135476173215056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173214672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173214288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173216592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173217168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173216976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173216400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173216016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173217552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173216208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135476173217936: Tensor

In [48]:
converter = tf.lite.TFLiteConverter.from_keras_model(mod)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_file='vrescnn_gpu_raf-db.tflite'
with open(tflite_file, 'wb') as f:
    f.write(tflite_model)

print("Conversia a fost realizată cu succes și modelul a fost salvat ca: ",tflite_file)

INFO:tensorflow:Assets written to: /tmp/tmpv6vkhz1f/assets


INFO:tensorflow:Assets written to: /tmp/tmpv6vkhz1f/assets


Conversia a fost realizată cu succes și modelul a fost salvat ca:  vrescnn_gpu_raf-db.tflite


W0000 00:00:1778165898.550737      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778165898.550765      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [50]:
interpreter = Interpreter(model_path="vrescnn_gpu_raf-db.tflite") 
interpreter.allocate_tensors()

In [51]:
import keras
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator


train_dir = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/train'
test_dir  = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test'

print("Versiune TensorFlow: ", tf.__version__)
print("Versiune Keras: ", keras.__version__)

img_size = 48
batch_size = 16
epoci = 50
num_classes = 7  # RAF-DB are 7 clase de bază

print("\nClase găsite:")
print(os.listdir(train_dir))

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.20
)

test_datagen = ImageDataGenerator(rescale=1./255)

print("Setul de antrenare (80%):")
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    subset='training'
)

print("Setul de validare (20%):")
val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
    subset='validation'
)

print("Setul de testare:")
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Versiune TensorFlow:  2.19.0
Versiune Keras:  3.10.0

Clase găsite:
['7', '2', '5', '3', '1', '4', '6']
Setul de antrenare (80%):
Found 9819 images belonging to 7 classes.
Setul de validare (20%):
Found 2452 images belonging to 7 classes.
Setul de testare:
Found 3068 images belonging to 7 classes.


In [52]:
import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

test_generator.reset()
y_true = test_generator.classes
predictions = []
durata_totala = 0

for img_batch, _ in test_generator:
    for img in img_batch:
        img_exp = np.expand_dims(img, axis=0).astype(np.float32)
        interpreter.set_tensor(input_index, img_exp)
        t1 = time.time()
        interpreter.invoke()
        t2 = time.time()
        durata_totala += (t2 - t1)
        output = interpreter.get_tensor(output_index)
        predictions.append(np.argmax(output))
    if len(predictions) >= len(y_true):
        break

predictions = predictions[:len(y_true)]
acc = sum(p == t for p, t in zip(predictions, y_true)) / len(y_true)
latenta = 1000 * durata_totala / len(y_true)

print(f"Acuratețe VRESCNN RAF-DB (.tflite): {acc*100:.2f}%")
print(f"Latența per imagine: {latenta:.4f} ms")
print(confusion_matrix(y_true, predictions))
etichete = list(test_generator.class_indices.keys())
print(classification_report(y_true, predictions, target_names=etichete))

Acuratețe VRESCNN RAF-DB (.tflite): 75.88%
Latența per imagine: 4.4819 ms
[[ 243    0    0   16    9   23   38]
 [  21    0    0    6   12   27    8]
 [   1    0    9   21   35   55   39]
 [   7    0    2 1086   25   18   47]
 [   2    0    9   28  363   23   53]
 [   4    0    2    5   12  131    8]
 [  21    0   12   36  104   11  496]]
              precision    recall  f1-score   support

           1       0.81      0.74      0.77       329
           2       0.00      0.00      0.00        74
           3       0.26      0.06      0.09       160
           4       0.91      0.92      0.91      1185
           5       0.65      0.76      0.70       478
           6       0.45      0.81      0.58       162
           7       0.72      0.73      0.72       680

    accuracy                           0.76      3068
   macro avg       0.54      0.57      0.54      3068
weighted avg       0.74      0.76      0.74      3068



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [54]:
import os

size_in_bytes = os.path.getsize('vrescnn_gpu_raf-db.tflite')
lite_sz = round(size_in_bytes/1000, 2)

print('Raport final Vrescnn raf-db:', 
      'Lite_sz', str(lite_sz)+'k',
      'acc:', str(round(100*acc, 2)),
      'lat', str(round(latenta, 2)))

Raport final Vrescnn raf-db: Lite_sz 404.18k acc: 75.88 lat 4.48
